In [26]:
import random
import pandas as pd
import json

random.seed(705)
n = 400
lokasi_list = ["Gudang Magelang", "Gudang Semarang", "Gudang Yogyakarta"]
kategori_dirty = ["sayur", "Sayur", "SAYUR", "buah", "Buah", "daging", "Daging"]

sopir_ids = [f"SP{i:03d}" for i in range(1, 25)]
lokasi_per_sopir = {pid: random.choice(lokasi_list) for pid in sopir_ids}

rows = []
for i in range(n):
    sopir_id = random.choice(sopir_ids)
    rows.append({
        "antar_id": f"ANT-{1000+i}",
        "sopir_id": sopir_id,
        "gudang_asal": lokasi_per_sopir[sopir_id],
        "kategori_produk": random.choice(kategori_dirty),
        "waktu_antar_jam": random.randint(2, 30),
        "biaya_antar": random.choice([20000, 30000, 40000, 50000, 60000]),
        "status_antar": random.choice(["Selesai", "Selesai", "Selesai", "Dibatalkan"]),
    })

rows += [rows[7], rows[28], rows[98]]
for idx in [9, 48, 155, 195]:
    rows[idx] = dict(rows[idx])
    rows[idx]["waktu_antar_jam"] = None
for idx in [14, 105]:
    rows[idx] = dict(rows[idx])
    rows[idx]["biaya_antar"] = None
for idx in [235, 285]:
    rows[idx] = dict(rows[idx])
    rows[idx]["waktu_antar_jam"] = -random.randint(1, 10)

pd.DataFrame(rows).to_csv("pengantaran.csv", index=False)

sopir_master = [
    {"sopir_id": pid, "nama_sopir": f"Sopir-{pid}",
    "gudang_asal": lokasi_per_sopir[pid],
    "rating_sopir": round(random.uniform(3.0,5.0),1)}
    for pid in sopir_ids
]
with open("sopir.json", "w") as f:
    for s in sopir_master:
        f.write(json.dumps(s) + "\n")

print("Data berhasil dibuat: pengantaran.csv dan sopir.json")

Data berhasil dibuat: pengantaran.csv dan sopir.json


SOAL 1. Ekstraksi Data melalui HDFS

In [31]:
# a. Unggah berkas pengantaran.csv ke HDFS pada direktori/user/[username]/uts/raw/

!hdfs -mkdir -f /user/azka/uts/raw/

!hdfs -put -f /pengantaran.csv/user/azka/uts/raw/

/home/azka/hadoop/libexec/hadoop-functions.sh: line 2417: HDFS_-MKDIR_USER: invalid variable name
ERROR: -mkdir is not COMMAND nor fully qualified CLASSNAME.
Usage: hdfs [OPTIONS] SUBCOMMAND [SUBCOMMAND OPTIONS]

  OPTIONS is none or any of:

--buildpaths                       attempt to add class files from build tree
--config dir                       Hadoop config directory
--daemon (start|status|stop)       operate on a daemon
--debug                            turn on shell script debug mode
--help                             usage information
--hostnames list[,of,host,names]   hosts to use in worker mode
--hosts filename                   list of hosts to use in worker mode
--loglevel level                   set the log4j level for this command
--workers                          turn on worker mode

  SUBCOMMAND is one of:


    Admin Commands:

cacheadmin           configure the HDFS cache
crypto               configure HDFS encryption zones
debug                run a Debug Admi

In [14]:
# b. Baca pengantar.csv dari HDFS menjadi spark DataFrame

pd.DataFrame(rows).to_csv("pengantaran.csv", index=False)

Soal 2. Eksplorasi dan Pembersihan Data

In [ ]:
# a. Identifikasi dan tampilkan bukti dari MINIMAL TIGA jenis masalah kualitas data yang berbeda pada pengantaran.csv.


Soal 3. Transformasi Data dan Penggabungan Masalah

In [ ]:
# c. Tambahkan kolom baru biaya_per_jam, dihitung dari biaya_antar dibagi waktu_anta

from pyspark.sql.functions import count(), sum as sprak_sum, avg



Soal 4. Analisis dengan Window Functions

In [ ]:
# a. Untuk SETIAP gudang_asal, tentukan TOP-3 sopir dengan jumlah pengantaran berkategori ‘Tepat Waktu’ TERBANYAK. Tampilkan gudang_asal, nama_sopir, dan jumlahnya.

from windows import Windows

window_space = Window("lokasi_asal") on="lokasi_per_sopir"
    .filter("Tepat_Waktu"),
    .agg("Tepat_Waktu" == 3)

window_space.show()

In [ ]:
# b. Untuk SETIAP gudang_asal, tentukan SATU sopir dengan rating TERTINGGI

df_gudang_asal = ("lokasi_asal") on="lokasi_per_sopir"
    .filter("Peringkat", row_number()),
    .agg("Peringkat" == 1)

window_space.show()

Soal 5. Analisis dengan SparkSQL

In [ ]:
from pyspark.sql import SparkSession

gudang_asal("""
    SELECT gudang_asal
    FROM lokasi_list
    GROUP BY total_pengantaran
    ORDER BY jumlah_terlambat
""")

gudang_asal.show()